# PHASE 1 — MACHINE LEARNING FOUNDATIONS


# Day 07 — Weekly Project: Customer Churn Prediction


## 1. Learning Objectives
By the end of this project, you will be able to:
- Integrate everything learned in Phase 1 (Splitting, Imputation, Scaling, Encoding, Pipelines).
- Establish a **Baseline Model** to serve as a minimum performance benchmark.
- Understand the concept of "Customer Churn" and why predicting it is a common real-world ML task.


## 2. Prerequisites
- Days 1 to 6 (The complete Scikit-learn foundational workflow).


## 3. Concept: The Baseline Model
A **Baseline** is the simplest possible model you can build. It could be predicting the majority class (e.g., predicting that *no one* churns), or it could be a basic Logistic Regression model with default parameters.

Why build a baseline? Because if you spend 3 weeks building a complex Deep Neural Network that achieves 85% accuracy, but a 5-minute Logistic Regression baseline achieves 84% accuracy... your complex model is practically useless.


## 4. Why Does This Matter?
Machine Learning is an iterative process. You never start by training the most complex model. 
1. Build a robust pipeline.
2. Establish a simple baseline.
3. Iterate and improve.
4. Ensure every complex addition actually provides a measurable return on investment (ROI) over the baseline.


## 5. Intuition
**Customer Churn**: A customer stops doing business with a company. 
If a Telecom company can predict *who* is going to cancel their subscription next month, they can proactively call them and offer a discount to stay. 
Our job today is to predict `Churn` (1 = Yes, 0 = No) based on customer profile data.


## 6. Mathematical Foundation (Dummy Classifiers)
Before we build a Machine Learning model, let's consider a Zero-Intelligence model. Let the dataset have $N$ samples, where $N_0$ is the number of retained customers and $N_1$ is the number of churned customers. 
If $N_0 > N_1$, a 'Most Frequent' dummy classifier will simply predict $0$ for every sample. 

Its accuracy will exactly equal: $\frac{N_0}{N_0 + N_1}$. If 80% of customers don't churn, predicting 'No' for everyone gives 80% accuracy! This is why accuracy can be misleading (a topic we will deep dive into in Phase 3).


## 7. The Project Setup
We will generate a synthetic but highly realistic Telecom Churn dataset. It will contain numerical data (Tenure, MonthlyCharges) and categorical data (InternetService, ContractType).


In [ ]:
import pandas as pd
import numpy as np

# Generate synthetic Churn Data
np.random.seed(42)
n_samples = 2000

data = {
    'Tenure_Months': np.random.randint(1, 72, n_samples),
    'Monthly_Charges': np.random.uniform(20.0, 120.0, n_samples),
    'Total_Charges': np.random.uniform(20.0, 8000.0, n_samples),
    'Internet_Service': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.3, 0.5, 0.2]),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.3, 0.2]),
    'Payment_Method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer'], n_samples)
}
df = pd.DataFrame(data)

# Introduce missing values realistically
df.loc[np.random.choice(df.index, 50, replace=False), 'Total_Charges'] = np.nan

# Create the target variable 'Churn' based on rules + noise
# Higher probability of churn if month-to-month and high charges
churn_prob = np.where(df['Contract'] == 'Month-to-month', 0.4, 0.05)
churn_prob += np.where(df['Monthly_Charges'] > 80, 0.2, 0.0)
churn_prob -= np.where(df['Tenure_Months'] > 40, 0.15, 0.0)
churn_prob = np.clip(churn_prob, 0.05, 0.85) # Keep probs realistic

df['Churn'] = np.random.binomial(1, churn_prob)

print(df.head())
print('\nMissing values:\n', df.isna().sum())
print('\nChurn Distribution:\n', df['Churn'].value_counts(normalize=True))


## 8. Identifying Features and Target
First, we separate `X` and `y`.


In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']


## 9. Train / Test Split
We split the data. Notice we use `stratify=y` because the target is imbalanced (~67% No, ~33% Yes).


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Training shape: {X_train.shape}, Test shape: {X_test.shape}')


## 10. Building the Preprocessing Pipeline
Now, identify which columns are numeric and which are categorical, and build the `ColumnTransformer`.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['Tenure_Months', 'Monthly_Charges', 'Total_Charges']
categorical_features = ['Internet_Service', 'Contract', 'Payment_Method']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numeric_features),
    ('cat', cat_pipeline, categorical_features)
])


## 11. Zero-Intelligence Baseline (Dummy Classifier)
Before running Logistic Regression, what happens if we just predict the majority class (No Churn) for everyone?


In [ ]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train, y_train)
dummy_acc = dummy_clf.score(X_test, y_test)

print(f'Dummy Classifier (Always predicts No Churn) Accuracy: {dummy_acc * 100:.2f}%')


> This means any machine learning model we build MUST score higher than this, otherwise it is useless!


## 12. Training the Baseline ML Model
Let's hook up our `preprocessor` to a `LogisticRegression` model.


In [ ]:
from sklearn.linear_model import LogisticRegression

ml_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=42))
])

ml_pipeline.fit(X_train, y_train)
ml_acc = ml_pipeline.score(X_test, y_test)
print(f'Logistic Regression Baseline Accuracy: {ml_acc * 100:.2f}%')


> We successfully beat the Dummy Classifier! The ML model is actually learning patterns.


## 13. Explaining Results (Error Analysis)
Let's see what the model is predicting compared to reality.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# We will cover this deeply in Phase 3, but let's take a sneak peek
ConfusionMatrixDisplay.from_estimator(ml_pipeline, X_test, y_test, cmap='Blues')
plt.title('Confusion Matrix (Baseline)')
plt.show()


> The matrix shows:
- True Positives (Churned and predicted Churn)
- True Negatives (Stayed and predicted Stayed)
- False Positives (Stayed but predicted Churn)
- False Negatives (Churned but predicted Stayed)


## 14. Phase 1 Evaluation
You have just completed an end-to-end Machine Learning pipeline. You:
1. Understood the problem (Churn).
2. Splitted the data safely.
3. Handled missing data.
4. Handled categorical strings.
5. Scaled numbers.
6. Prevented data leakage using Pipelines.
7. Established a Dummy baseline.
8. Beat the baseline with a linear model.


## 15. Real-World Application
In the real world, the data engineering team would schedule this pipeline to run every night on active customers. Anyone predicted as '1' (Churn) would be automatically emailed a 20% discount coupon the next morning.


## 16. Capstone Exercise for Phase 1
Your turn! The HR department wants to predict Employee Attrition (whether an employee will quit). 
They gave you this dataframe. Build the full pipeline and baseline model.


In [ ]:
# YOUR CODE HERE
hr_df = pd.DataFrame({
    'Age': [28, 45, 32, 50, 25, 40],
    'Department': ['Sales', 'IT', 'Sales', 'HR', 'IT', 'IT'],
    'Distance_From_Home': [2.5, 15.0, np.nan, 5.0, 25.0, 1.0],
    'Quits': [1, 0, 1, 0, 1, 0]
})

# Write your complete pipeline to predict 'Quits' below:
X_hr = hr_df.drop('Quits', axis=1)
y_hr = hr_df['Quits']

hr_num = ['Age', 'Distance_From_Home']
hr_cat = ['Department']

hr_pre = ColumnTransformer([
    ('n', Pipeline([('imp', SimpleImputer(strategy='mean')), ('scl', StandardScaler())]), hr_num),
    ('c', OneHotEncoder(), hr_cat)
])

hr_pipe = Pipeline([('pre', hr_pre), ('clf', LogisticRegression())])
hr_pipe.fit(X_hr, y_hr)
print('HR Model Trained successfully on tiny dataset. Score:', hr_pipe.score(X_hr, y_hr))


## 17. Common Mistakes (Phase 1 Review)
- Memorize this: **NEVER fit your Scaler or Imputer on the Test Set!** Use Pipelines to guarantee you don't mess this up.
- Never assume a 90% accuracy is good without checking the target distribution. If 99% of people don't click an ad, a model that predicts "No Click" for everyone is 99% accurate but completely useless.


## 18. Interview Questions
- **Beginner**: What are the components of a Scikit-learn Pipeline?
- **Intermediate**: Why is a Dummy Classifier important?
- **Advanced**: In our churn pipeline, what would happen if we used `SimpleImputer` *after* the `StandardScaler`?


## 19. Knowledge Check
- What estimator always predicts the most frequent class? (`DummyClassifier`)


## 20. Summary of Phase 1
You now know the structural mechanics of Scikit-learn. In **Phase 2**, we will move beyond Logistic Regression and dive deep into the mathematics and algorithms behind powerful ML models (Decision Trees, Random Forests, Gradient Boosting) for Regression problems!


## 21. Homework / Phase 1 Assessment
Take a break! Review the concepts from Days 1-7. Tomorrow we begin Regression modeling.
